In [3]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, IterativeImputer

import causalpype as cp
from causalpype import CausalModel
from causalpype.tasks import (
    ArrowStrength,
    ATE,
    CausalEffectCurve,
    DistributionChange,
    FairnessAudit,
)
from causalpype.plotting import plot_arrow_strength, plot_causal_effect_curve

In [4]:
import dowhy.gcm as gcm
import networkx as nx
import pandas as pd

# Build graph manually
g = nx.DiGraph([("age","sysBP"), ("age","CHD"),
                ("sysBP","CHD")])
scm = gcm.InvertibleStructuralCausalModel(g)

# --- Load data --------------------------------------------------------------
df = pd.read_csv('data/framingham.csv')
print(f'Raw shape: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df.head(3)

# --- Preprocessing -----------------------------------------------------------
# Binary/categorical columns: mode imputation
cat_cols = ['male', 'education', 'currentSmoker', 'BPMeds',
            'prevalentStroke', 'prevalentHyp', 'diabetes']

cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

# Continuous columns: MICE (IterativeImputer)
cont_cols = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP',
             'BMI', 'heartRate', 'glucose']

df_for_mice = df.copy()
for col in cat_cols:
    df_for_mice[col] = df_for_mice[col].astype(float)

mice = IterativeImputer(max_iter=20, random_state=42, sample_posterior=False)
imputed = pd.DataFrame(
    mice.fit_transform(df_for_mice),
    columns=df_for_mice.columns,
    index=df_for_mice.index,
)

# Copy imputed continuous values back
for col in cont_cols:
    df[col] = imputed[col]

# Plausibility clipping
df['cigsPerDay'] = df['cigsPerDay'].clip(lower=0, upper=70)
df['totChol']    = df['totChol'].clip(lower=100, upper=600)
df['sysBP']      = df['sysBP'].clip(lower=80, upper=300)
df['diaBP']      = df['diaBP'].clip(lower=40, upper=200)
df['BMI']        = df['BMI'].clip(lower=10, upper=60)
df['heartRate']  = df['heartRate'].clip(lower=30, upper=220)
df['glucose']    = df['glucose'].clip(lower=40, upper=500)

# Keep TenYearCHD as numeric
df_clean = df.copy()
print(f'Clean shape: {df_clean.shape}')
print(f'CHD prevalence: {df_clean["TenYearCHD"].mean():.1%}')
print(f'Remaining NaNs: {df_clean.isnull().sum().sum()}')

Raw shape: (4240, 16)
Missing values:
education     105
cigsPerDay     29
BPMeds         53
totChol        50
BMI            19
heartRate       1
glucose       388
dtype: int64
Clean shape: (4240, 16)
CHD prevalence: 15.2%
Remaining NaNs: 0


In [5]:
# --- DAG & Model ------------------------------------------------------------
dag = {
    'age':             ['sysBP', 'diaBP', 'totChol', 'glucose', 'BMI',
                        'heartRate', 'TenYearCHD'],
    'male':            ['sysBP', 'diaBP', 'totChol', 'BMI', 'heartRate',
                        'currentSmoker', 'TenYearCHD'],
    'education':       ['currentSmoker'],
    'currentSmoker':   ['cigsPerDay'],
    'cigsPerDay':      ['heartRate', 'TenYearCHD'],
    'BMI':             ['sysBP', 'diaBP', 'totChol', 'glucose',
                        'diabetes', 'TenYearCHD'],
    'sysBP':           ['prevalentHyp', 'TenYearCHD'],
    'diaBP':           ['prevalentHyp'],
    'prevalentHyp':    ['BPMeds'],
    'glucose':         ['diabetes'],
    'diabetes':        ['TenYearCHD'],
    'totChol':         ['TenYearCHD'],
    'heartRate':       ['TenYearCHD'],
    'prevalentStroke': ['TenYearCHD'],
}

model = CausalModel(dag, assignment_quality='better')
model.fit(df_clean)

print(f'Nodes: {model.graph.number_of_nodes()}')
print(f'Edges: {model.graph.number_of_edges()}')

Fitting causal mechanism of node BPMeds: 100%|██████████| 16/16 [00:00<00:00, 61.85it/s]         

Nodes: 16
Edges: 33


In [11]:
import numpy as np
import dowhy.gcm as gcm
import networkx as nx

np.random.seed(42)

g = nx.DiGraph([("age","sysBP"), ("age","TenYearCHD"),
                ("sysBP","TenYearCHD")])
scm = gcm.InvertibleStructuralCausalModel(g)

# Assign mechanisms and fit -- required before any task
gcm.auto.assign_causal_mechanisms(
    scm, df_clean,
    quality=gcm.auto.AssignmentQuality.BETTER)
gcm.fit(scm, df_clean)

# Task 1: ATE -- separate function, its own signature
ate = gcm.average_causal_effect(
    scm, target_node="TenYearCHD",
    interventions_alternative={"sysBP": lambda x: 140},
    interventions_reference={"sysBP": lambda x: 120},
    num_samples_to_draw=2000)
print(f"ATE: {ate:.4f}")

# Task 2: arrow strength -- different function, different args
strengths = gcm.arrow_strength(scm, target_node="TenYearCHD")
for edge, val in strengths.items():
    print(f"{edge}: {val:.4f}")

Fitting causal mechanism of node TenYearCHD: 100%|██████████| 3/3 [00:00<00:00, 145.69it/s]


ATE: 0.0140
('age', 'TenYearCHD'): 0.0009
('sysBP', 'TenYearCHD'): 0.0017


In [12]:
import numpy as np
from causalpype import CausalModel, ATE, ArrowStrength

np.random.seed(42)

model = CausalModel({
    'age':   ['sysBP', 'TenYearCHD'],
    'sysBP': ['TenYearCHD'],
}, assignment_quality='better')
model.fit(df_clean)

results = model.run([
    ATE(treatment='sysBP', outcome='TenYearCHD',
        treatment_value=140, control_value=120),
    ArrowStrength(target='TenYearCHD'),
])
for r in results:
    print(r)

Fitting causal mechanism of node TenYearCHD: 100%|██████████| 3/3 [00:00<00:00, 159.59it/s]


                       ATE Results                        
 Treatment                                           sysBP
 Outcome                                        TenYearCHD
 Treatment Value                                       140
 Control Value                                         120
----------------------------------------------------------
 Estimate                                           0.0140
 Num Samples                                         2,000
                  Arrow Strength Results                  
 Target                                         TenYearCHD
----------------------------------------------------------
  sysBP -> TenYearCHD                               0.0017
  age -> TenYearCHD                                 0.0009
